# ⚡ AETHER All-in-One Studio — Google Colab & Kaggle Runner

**Models:** Stable Diffusion XL (Images) + Fish Audio S2 Pro (Voice)

This notebook runs **BOTH** models simultaneously on a single free T4 GPU using a unified API tunnel and CPU offloading!

> ⚠️ **Enable GPU before running!**  
> `Runtime → Change runtime type → T4 GPU`

In [ ]:
# 1. Install Dependencies & Clone Models
# --- Environment + storage detection (Colab vs Kaggle vs local) ---
# Kaggle has no /content, and caps /kaggle/working at 20 GB. The S2 Pro model
# does not fit there via git-lfs (which stores every file twice), so the model
# goes to whichever mount actually has free space and is symlinked in.
import os, shutil

if os.path.isdir('/kaggle'):
    BASE = '/kaggle/working'
elif os.path.isdir('/content'):
    BASE = '/content'
else:
    BASE = os.getcwd()
FISH_DIR = os.path.join(BASE, 'fish-speech')
os.makedirs(BASE, exist_ok=True)

# Pick the roomiest scratch mount for the ~10 GB of model weights.
_cands = []
for _c in ('/kaggle/temp', '/tmp', os.path.join(BASE, '.scratch')):
    try:
        os.makedirs(_c, exist_ok=True)
        _cands.append((shutil.disk_usage(_c).free, _c))
    except Exception:
        pass
SCRATCH = max(_cands)[1] if _cands else BASE
os.environ['HF_HOME'] = os.path.join(SCRATCH, 'hf')

def disk_report(label=''):
    print(f'💾 Disk {label}')
    for _p in dict.fromkeys([BASE, SCRATCH, '/']):
        try:
            _t, _u, _f = shutil.disk_usage(_p)
            print(f'   {_p:<24} {_f/1e9:6.1f} GB free / {_t/1e9:6.1f} GB total')
        except Exception:
            pass

print(f'📁 Environment base: {BASE}')
print(f'📁 Fish Speech dir : {FISH_DIR}')
print(f'📁 Model scratch   : {SCRATCH}')

disk_report('before install')

os.chdir(BASE)

# Remove any half-finished previous attempt so a re-run starts clean and frees
# the space that a failed run left behind.
!rm -rf "{FISH_DIR}"
!pip cache purge 2>/dev/null || true

!apt-get update -qq
!apt-get install -y portaudio19-dev build-essential rustc cargo git git-lfs psmisc

!git clone https://github.com/fishaudio/fish-speech.git "{FISH_DIR}"
os.chdir(FISH_DIR)
print(f'✅ Working directory is now: {os.getcwd()}')

# --- APPLY MEMORY OOM FIX ---
# Force bfloat16 at init: PyTorch's float32 default needs ~20 GB and crashes a
# free T4/P100. bfloat16 halves it.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
llama_path = os.path.join(FISH_DIR, "fish_speech/models/text2semantic/llama.py")
with open(llama_path, "r") as f:
    code = f.read()

code = code.replace(
    "model = model_cls(config)",
    "torch.set_default_dtype(torch.bfloat16)\n        model = model_cls(config)\n        torch.set_default_dtype(torch.float32)"
)

# Restrict max sequence length (saves ~4.5 GB of KV-cache VRAM)
code = code.replace(
    "config = BaseModelArgs.from_pretrained(str(path))",
    "config = BaseModelArgs.from_pretrained(str(path))\n        config.max_seq_len = 3072"
)
with open(llama_path, "w") as f:
    f.write(code)
print("✅ Out-of-Memory (OOM) fix successfully applied to Fish Speech source code!")
# ----------------------------

# Upgrade pip to ensure it pulls binary wheels instead of building from source
!python -m pip install -q --upgrade pip wheel
!pip install -q tokenizers transformers "huggingface_hub>=0.23"

# Fix protobuf/tensorflow crash on Kaggle by removing tensorflow (we only use PyTorch!)
!pip uninstall -y tensorflow
!pip install -q -U protobuf

# Install safely and FORCE torchvision downgrade to match Fish Speech's PyTorch version
!pip install -q -e . torchvision
!pip install -q accelerate torch fastapi uvicorn httpx pyngrok nest_asyncio pyrootutils psutil

# --- DOWNLOAD THE S2 PRO MODEL (single copy) ---
# NOT `git clone` — git-lfs keeps the blob in .git/lfs/objects *and* in the
# working tree, doubling ~10 GB of weights and exhausting Kaggle's 20 GB quota.
from huggingface_hub import snapshot_download

# cache_dir is passed explicitly: huggingface_hub resolves HF_HOME at import
# time, so the env var alone is unreliable if transformers was already imported.
HF_CACHE = os.path.join(SCRATCH, "hf")
print(f"⬇️  Downloading fishaudio/s2-pro into {HF_CACHE} (single copy)...")
model_path = snapshot_download(repo_id="fishaudio/s2-pro", cache_dir=HF_CACHE)
print(f"✅ Model at: {model_path}")

ckpt_link = os.path.join(FISH_DIR, "checkpoints", "s2-pro")
os.makedirs(os.path.dirname(ckpt_link), exist_ok=True)
if os.path.islink(ckpt_link):
    os.unlink(ckpt_link)
elif os.path.isdir(ckpt_link):
    shutil.rmtree(ckpt_link)
os.symlink(model_path, ckpt_link)
print(f"🔗 Linked {ckpt_link} -> {model_path}")

_codec = os.path.join(ckpt_link, "codec.pth")
print(f"🔍 codec.pth present: {os.path.exists(_codec)}")
if not os.path.exists(_codec):
    print("⚠️  codec.pth missing — the API server will fail to start.")
# ----------------------------

print("✅ Dependencies and Model installed!")

# --- APPLY VRAM LEAK FIX ---
views_path = os.path.join(FISH_DIR, "tools/server/views.py")
with open(views_path, "r") as f:
    vcode = f.read()

vcode = vcode.replace(
    "return StreamingResponse(generator(), media_type=\"audio/wav\")",
    "def cache_clearing_generator():\n        for chunk in generator():\n            yield chunk\n        torch.cuda.empty_cache()\n    return StreamingResponse(cache_clearing_generator(), media_type=\"audio/wav\")"
)
with open(views_path, "w") as f:
    f.write(vcode)
print("✅ VRAM leak patch applied to views.py!")
# ----------------------------

disk_report('after install')

In [ ]:
# 2. Authenticate ngrok
# REPLACE "YOUR_TOKEN_HERE" WITH YOUR ACTUAL NGROK TOKEN IF SECRETS ARE NOT WORKING
MANUAL_TOKEN = ""

try:
    from google.colab import userdata
    colab_env = True
except ImportError:
    colab_env = False

try:
    if MANUAL_TOKEN:
        ngrok_token = MANUAL_TOKEN
    elif colab_env:
        ngrok_token = userdata.get("NGROK_TOKEN")
    else:
        # For Kaggle or other envs without Colab userdata
        import os
        ngrok_token = os.environ.get("NGROK_TOKEN", "")
        
    if not ngrok_token:
        raise ValueError("Token is empty!")
        
    !ngrok authtoken {ngrok_token}
    print("✅ ngrok authenticated")
except Exception as e:
    print("\n❌ FATAL ERROR: Could not authenticate with ngrok!")
    print("You have two options to fix this:")
    print("1. Paste your token between the quotes in MANUAL_TOKEN = \"\" at the top of this cell.")
    print("2. OR Add your ngrok token to Colab Secrets (the 🔑 icon on the left) as NGROK_TOKEN and turn the toggle switch ON.")
    raise Exception("STOPPING: You must provide a valid ngrok token before continuing!")

# 3. Add Custom Voices (Zero-Shot Cloning)
Fish Speech S2 Pro allows you to instantly clone ANY voice just by providing a 10-second reference audio file!

**How to clone your own voice:**
1. In the file explorer on the left, navigate to the `references/` folder inside your `fish-speech` directory (`/content/fish-speech/references/` on Colab, `/kaggle/working/fish-speech/references/` on Kaggle — the install cell prints the exact path)
2. Create a new folder with your voice name (e.g. `My_Voice`)
3. Upload a 10-20s clean `.wav` or `.mp3` file of the voice speaking and name it `audio.wav`
4. Create a text file called `audio.lab` in the same folder and type out exactly what is being said in the audio.
5. Restart the Unified API Server cell below so it detects the new folder!

*(Run the cell below to automatically install a sample "JFK Presidential" voice pack to test it out!)*

In [ ]:
import requests
# --- Environment + storage detection (Colab vs Kaggle vs local) ---
# Kaggle has no /content, and caps /kaggle/working at 20 GB. The S2 Pro model
# does not fit there via git-lfs (which stores every file twice), so the model
# goes to whichever mount actually has free space and is symlinked in.
import os, shutil

if os.path.isdir('/kaggle'):
    BASE = '/kaggle/working'
elif os.path.isdir('/content'):
    BASE = '/content'
else:
    BASE = os.getcwd()
FISH_DIR = os.path.join(BASE, 'fish-speech')
os.makedirs(BASE, exist_ok=True)

# Pick the roomiest scratch mount for the ~10 GB of model weights.
_cands = []
for _c in ('/kaggle/temp', '/tmp', os.path.join(BASE, '.scratch')):
    try:
        os.makedirs(_c, exist_ok=True)
        _cands.append((shutil.disk_usage(_c).free, _c))
    except Exception:
        pass
SCRATCH = max(_cands)[1] if _cands else BASE
os.environ['HF_HOME'] = os.path.join(SCRATCH, 'hf')

def disk_report(label=''):
    print(f'💾 Disk {label}')
    for _p in dict.fromkeys([BASE, SCRATCH, '/']):
        try:
            _t, _u, _f = shutil.disk_usage(_p)
            print(f'   {_p:<24} {_f/1e9:6.1f} GB free / {_t/1e9:6.1f} GB total')
        except Exception:
            pass

print(f'📁 Environment base: {BASE}')
print(f'📁 Fish Speech dir : {FISH_DIR}')
print(f'📁 Model scratch   : {SCRATCH}')


if not os.path.isdir(FISH_DIR):
    raise RuntimeError(f'{FISH_DIR} does not exist — run the install cell (cell 1) first.')

_free_mb = shutil.disk_usage(FISH_DIR).free / 1e6
if _free_mb < 200:
    disk_report('current')
    raise RuntimeError(
        f'Only {_free_mb:.0f} MB free on {FISH_DIR} — not enough for the voice pack. '
        'The model must not live under the working-directory quota; re-run cell 1, '
        'which downloads it to scratch and symlinks it in.'
    )

print("🎙️ Downloading AETHERSTUDIO Premium Voice Pack (10 Ultra-Quality Human Voices)...")

voices = {
    "JFK_Presidential": {
        "url": "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main/jfk.wav",
        "text": "And so my fellow Americans, ask not what your country can do for you, ask what you can do for your country."
    },
    "MLK_Historical": {
        "url": "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main/mlk.wav",
        "text": "I have a dream that one day this nation will rise up and live out the true meaning of its creed."
    },
    "Professional_Female": {
        "url": "https://raw.githubusercontent.com/coqui-ai/TTS/dev/tests/data/ljspeech/wavs/LJ001-0001.wav",
        "text": "Printing, in the only sense with which we are at present concerned, differs from most if not from all the arts and crafts represented in the Exhibition"
    },
    "XTTS_Male_1": {
        "url": "https://huggingface.co/spaces/coqui/xtts/resolve/main/examples/male.wav",
        "text": "When I wake up, I expect a coffee ready and waiting for me."
    },
    "XTTS_Female_1": {
        "url": "https://huggingface.co/spaces/coqui/xtts/resolve/main/examples/female.wav",
        "text": "This is a great day to learn something new about artificial intelligence."
    },
    "TED_Talk_Speaker": {
        "url": "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main/ted_60_16k.wav",
        "text": "Today I want to share with you a story about the future of technology."
    },
    "OpenVoice_Speaker_0": {
        "url": "https://raw.githubusercontent.com/myshell-ai/OpenVoice/main/resources/demo_speaker0.mp3",
        "text": "I am a high quality human reference voice used for cloning."
    },
    "OpenVoice_Speaker_1": {
        "url": "https://raw.githubusercontent.com/myshell-ai/OpenVoice/main/resources/demo_speaker1.mp3",
        "text": "I am a high quality human reference voice used for cloning."
    },
    "OpenVoice_Speaker_2": {
        "url": "https://raw.githubusercontent.com/myshell-ai/OpenVoice/main/resources/demo_speaker2.mp3",
        "text": "I am a high quality human reference voice used for cloning."
    },
    "French_Speaker": {
        "url": "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main/french-audio.wav",
        "text": "Bonjour, je suis une voix française de haute qualité."
    }
}

for voice_id, data in voices.items():
    os.makedirs(f"{FISH_DIR}/references/{voice_id}", exist_ok=True)
    wav_path = f"{FISH_DIR}/references/{voice_id}/audio.wav"
    if not os.path.exists(wav_path):
        print(f"Downloading {voice_id}...")
        with open(wav_path, "wb") as f:
            f.write(requests.get(data["url"]).content)
    
    lab_path = f"{FISH_DIR}/references/{voice_id}/audio.lab"
    with open(lab_path, "w", encoding="utf-8") as f:
        f.write(data["text"])
    
    print(f"✅ {voice_id} installed!")

print("🎉 All 10 voices installed! Restart the API Server cell below!")

# Verify what the API server will actually discover (it reads ./references
# relative to its own working directory — which must be FISH_DIR).
ref_root = os.path.join(FISH_DIR, "references")
found = sorted(
    d for d in os.listdir(ref_root)
    if os.path.isdir(os.path.join(ref_root, d))
) if os.path.isdir(ref_root) else []
print(f"\n🔍 References directory: {ref_root}")
print(f"🔍 Voices the server will expose ({len(found)}): {found}")
if not found:
    print("❌ No voices found! The API will return an empty voice list.")


In [ ]:
# 4. Launch Unified API Server
import nest_asyncio
import uvicorn
import torch
import httpx
import subprocess
import time
import requests
import psutil
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from starlette.background import BackgroundTask
from pyngrok import ngrok

# --- Environment + storage detection (Colab vs Kaggle vs local) ---
# Kaggle has no /content, and caps /kaggle/working at 20 GB. The S2 Pro model
# does not fit there via git-lfs (which stores every file twice), so the model
# goes to whichever mount actually has free space and is symlinked in.
import os, shutil

if os.path.isdir('/kaggle'):
    BASE = '/kaggle/working'
elif os.path.isdir('/content'):
    BASE = '/content'
else:
    BASE = os.getcwd()
FISH_DIR = os.path.join(BASE, 'fish-speech')
os.makedirs(BASE, exist_ok=True)

# Pick the roomiest scratch mount for the ~10 GB of model weights.
_cands = []
for _c in ('/kaggle/temp', '/tmp', os.path.join(BASE, '.scratch')):
    try:
        os.makedirs(_c, exist_ok=True)
        _cands.append((shutil.disk_usage(_c).free, _c))
    except Exception:
        pass
SCRATCH = max(_cands)[1] if _cands else BASE
os.environ['HF_HOME'] = os.path.join(SCRATCH, 'hf')

def disk_report(label=''):
    print(f'💾 Disk {label}')
    for _p in dict.fromkeys([BASE, SCRATCH, '/']):
        try:
            _t, _u, _f = shutil.disk_usage(_p)
            print(f'   {_p:<24} {_f/1e9:6.1f} GB free / {_t/1e9:6.1f} GB total')
        except Exception:
            pass

print(f'📁 Environment base: {BASE}')
print(f'📁 Fish Speech dir : {FISH_DIR}')
print(f'📁 Model scratch   : {SCRATCH}')

if not os.path.isdir(FISH_DIR):
    raise RuntimeError(
        f'{FISH_DIR} does not exist — run the install cell (cell 1) first.'
    )
os.chdir(FISH_DIR)

nest_asyncio.apply()

# Ensure any zombie ngrok tunnels from previous interrupted runs are killed
ngrok.kill()
os.system("killall -9 ngrok 2>/dev/null")
os.system("pkill -9 -f \"tools.api_server\" || true")
for conn in psutil.net_connections():
    if conn.laddr.port in [8000, 8081] and conn.status == 'LISTEN':
        try:
            psutil.Process(conn.pid).terminate()
        except:
            pass
time.sleep(1)

# Show which voices exist BEFORE boot — the server only scans ./references once,
# relative to its own cwd, so a mismatch here is why the app shows no voices.
ref_root = os.path.join(FISH_DIR, "references")
found = sorted(
    d for d in os.listdir(ref_root)
    if os.path.isdir(os.path.join(ref_root, d))
) if os.path.isdir(ref_root) else []
print(f"🎙️  Reference voices detected in {ref_root}: {found if found else 'NONE'}")
if not found:
    print("⚠️  No reference voices — run the voice-pack cell above, then re-run THIS cell.")

print("🐟 Starting Fish Speech S2 Pro API Server (Subprocess)...")
fish_process = subprocess.Popen(
    ["python", "-m", "tools.api_server", "--listen", "127.0.0.1:8081", "--half"],
    cwd=FISH_DIR  # CRITICAL: references/ is resolved relative to this directory
)

print("⏳ Waiting for Fish Speech to boot (Takes ~2 mins)...")
while True:
    if fish_process.poll() is not None:
        raise RuntimeError(
            f"Fish Speech server exited early with code {fish_process.returncode}. "
            "Scroll up for its traceback."
        )
    try:
        if requests.get("http://127.0.0.1:8081/v1/health", timeout=5).status_code == 200:
            print("✅ Fish Speech Ready!")
            break
    except:
        pass
    time.sleep(5)

try:
    _refs = requests.get(
        "http://127.0.0.1:8081/v1/references/list?format=json", timeout=10
    ).json()
    print(f"✅ API reports these voices: {_refs.get('reference_ids')}")
except Exception as e:
    print(f"⚠️  Could not read voice list from the API: {e}")


app = FastAPI(title="AETHER All-in-One")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

client = httpx.AsyncClient(base_url="http://127.0.0.1:8081", timeout=None)

# Headers that describe a single hop's framing/encoding. They MUST NOT be copied
# between the upstream response and our own — Starlette sets its own framing for a
# StreamingResponse, so passing the upstream Content-Length / Transfer-Encoding
# through makes browsers truncate or reject the audio stream.
HOP_BY_HOP = {
    "connection", "keep-alive", "proxy-authenticate", "proxy-authorization",
    "te", "trailers", "transfer-encoding", "upgrade",
    "content-length", "content-encoding",
}


@app.get("/aether/status")
async def aether_status():
    """Quick diagnostics you can open directly in a browser."""
    try:
        r = await client.get("/v1/references/list?format=json")
        voices = r.json().get("reference_ids", [])
    except Exception as e:
        voices = f"error: {e}"
    return JSONResponse({
        "ok": True,
        "base": BASE,
        "fish_dir": FISH_DIR,
        "cwd": os.getcwd(),
        "references_dir": ref_root,
        "voices": voices,
    })


@app.api_route("/v1/{path:path}", methods=["GET", "POST", "PUT", "DELETE", "OPTIONS"])
async def proxy_fish(path: str, request: Request):
    url = httpx.URL(path=request.url.path, query=request.url.query.encode("utf-8"))
    # Forward the client's headers minus framing ones, and ask upstream for an
    # unencoded body so the bytes we stream out match the headers we send.
    fwd = {
        k: v for k, v in request.headers.items()
        if k.lower() not in HOP_BY_HOP and k.lower() not in ("host", "accept-encoding")
    }
    fwd["accept-encoding"] = "identity"

    req = client.build_request(
        request.method, url, headers=fwd, content=await request.body()
    )
    res = await client.send(req, stream=True)

    safe_headers = {
        k: v for k, v in res.headers.items() if k.lower() not in HOP_BY_HOP
    }
    return StreamingResponse(
        res.aiter_raw(),
        status_code=res.status_code,
        headers=safe_headers,
        media_type=res.headers.get("content-type"),
        # Without this the upstream response is never released and the connection
        # pool leaks until the tunnel stops answering.
        background=BackgroundTask(res.aclose),
    )

public_url = ngrok.connect(8000).public_url
print("\n" + "="*60)
print("🚀 AETHER ALL-IN-ONE API IS LIVE")
print("="*60)
print(f"  Paste this ONE link into the FISH SPEECH Settings box in AetherStudio:")
print(f"  URL: {public_url}")
print(f"  Sanity check in your browser: {public_url}/aether/status")
print("="*60)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
import asyncio
await server.serve()